In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import fastf1
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error

import warnings
warnings.filterwarnings("ignore")

In [2]:
# Enable FastF1 caching
import os
os.makedirs("f1_cache", exist_ok=True)
fastf1.Cache.enable_cache("f1_cache")

### 2025 Data

In [3]:
# Load 2025 Italian GP race session, lap and sector times
session_2025 = fastf1.get_session(2026, "Italy", "R")
session_2025.load()
laps_2025 = session_2025.laps[["Driver", "LapTime", "Sector1Time", "Sector2Time", "Sector3Time"]].copy()
laps_2025.dropna(inplace=True)

core           INFO 	Loading data for Italian Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Driver 12 completed the race distance 00:00.013000 before the recorded end of the session.
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 22 drivers: ['12', '63', '3', '1', '81', '44', '10', '41', '43', '22', '5', '27

In [4]:
# Convert times to seconds
for col in ["LapTime", "Sector1Time", "Sector2Time", "Sector3Time"]:
    laps_2025[f"{col} (s)"] = laps_2025[col].dt.total_seconds()

In [5]:
# Group by driver to get average sector times per driver
sector_times_2025 = laps_2025.groupby("Driver")[["Sector1Time (s)", "Sector2Time (s)", "Sector3Time (s)"]].mean().reset_index()

### Quali Data 2026

In [6]:
# Qualifying data for 2026 Italian GP (Monza)
# Time = lap from the last session each driver reached (Q3 for P1-10, Q2 for P11-16, Q1 for P17-22)
qualifying_2026 = pd.DataFrame({
    "Driver": ["GAS", "RUS", "PIA", "LEC", "HAM", "VER", "ANT", "COL", "NOR", "LIN",
               "BOR", "BEA", "HUL", "LAW", "SAI", "OCO",
               "TSU", "ALB", "BOT", "PER", "ALO", "STR"],
    "QualifyingTime (s)": [81.786, 81.846, 81.966, 82.004, 82.011, 82.070, 82.093, 82.220, 82.256, 82.286,
                           82.517, 82.756, 82.779, 82.821, 83.453, 83.454,
                           83.755, 84.356, 84.364, 84.595, 85.150, 85.222]
})

### Weather component

In [7]:
API_KEY = "PUT YOUR API HERE"
# Monza (Autodromo Nazionale Monza), Italy -- 2026 Italian GP venue
weather_url = f"http://api.openweathermap.org/data/2.5/forecast?lat=45.6156&lon=9.2811&appid={API_KEY}&units=metric"
response = requests.get(weather_url)
weather_data = response.json()
print(weather_data)

{'cod': '200', 'message': 0, 'cnt': 40, 'list': [{'dt': 1791817200, 'main': {'temp': 18.85, 'feels_like': 18.57, 'temp_min': 18.85, 'temp_max': 20.04, 'pressure': 1018, 'sea_level': 1018, 'grnd_level': 994, 'humidity': 68, 'temp_kf': -1.19, 'dew_point': 12.82}, 'weather': [{'id': 800, 'main': 'Clear', 'description': 'clear sky', 'icon': '01d'}], 'clouds': {'all': 8}, 'wind': {'speed': 2.45, 'deg': 130, 'gust': 5.71}, 'visibility': 10000, 'pop': 0, 'sys': {'pod': 'd'}, 'dt_txt': '2026-10-12 15:00:00'}, {'dt': 1791828000, 'main': {'temp': 18.43, 'feels_like': 18.13, 'temp_min': 17.59, 'temp_max': 18.43, 'pressure': 1019, 'sea_level': 1019, 'grnd_level': 996, 'humidity': 69, 'temp_kf': 0.84, 'dew_point': 12.64}, 'weather': [{'id': 800, 'main': 'Clear', 'description': 'clear sky', 'icon': '01n'}], 'clouds': {'all': 7}, 'wind': {'speed': 1.83, 'deg': 95, 'gust': 3.69}, 'visibility': 10000, 'pop': 0, 'sys': {'pod': 'n'}, 'dt_txt': '2026-10-12 18:00:00'}, {'dt': 1791838800, 'main': {'temp': 1

In [8]:
# Extract the relevant weather data for the race (Sunday at 3pm local time)
forecast_time = "2026-09-06 15:00:00"
forecast_data = None
for forecast in weather_data["list"]:
    if forecast["dt_txt"] == forecast_time:
        forecast_data = forecast
        break

if forecast_data:
    rain_probability = forecast_data["pop"]
    temperature = forecast_data["main"]["temp"]
else:
    rain_probability = 0
    temperature = 20

In [9]:
# Merge qualifying data with sector times data
merged_data = qualifying_2026.merge(sector_times_2025, left_on="Driver", right_on="Driver", how="left")

In [10]:
# Create weather features for the model
merged_data["RainProbability"] = rain_probability
merged_data["Temperature"] = temperature

In [11]:
# Attach average lap time to merged_data the same way sector times were attached (left join),
# so a driver with no clean lap (e.g. retired, like LEC) is dropped consistently from X and y.
merged_data["LapTime (s)"] = merged_data["Driver"].map(laps_2025.groupby("Driver")["LapTime (s)"].mean())
merged_data.dropna(subset=["LapTime (s)"], inplace=True)

# Define feature set (Qualifying + Sector Times + Weather)
X = merged_data[["QualifyingTime (s)", "Sector1Time (s)", "Sector2Time (s)", "Sector3Time (s)", "RainProbability", "Temperature"]].fillna(0)

# Use average lap time as the target
y = merged_data["LapTime (s)"]

In [12]:
# Train Gradient Boosting Model
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=38)
model = GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, random_state=38)
model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",200
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random seed given to each Tree estimator at eachboosting iteration.In addition, it controls the random permutation of the features ateach split (see Notes for more details).It also controls the random splitting of the training data to obtain avalidation set if `n_iter_no_change` is not None.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",38
,"loss loss: {'squared_error', 'absolute_error', 'huber', 'quantile'}, default='squared_error'Loss function to be optimized. 'squared_error' refers to the squarederror for regression. 'absolute_error' refers to the absolute error ofregression and is a robust loss function. 'huber' is acombination of the two. 'quantile' allows quantile regression (use`alpha` to specify the quantile).See:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_quantile.py`for an example that demonstrates quantile regression for creatingprediction intervals with `loss='quantile'`.",'squared_error'
,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.",0.1
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1.0
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'This parameter has no effect... versionadded:: 0.18.. deprecated:: 1.9 `criterion` is deprecated and will be removed in 1.11.",'deprecated'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",3
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf

In [13]:
# Predict race times using 2026 qualifying and sector data
predicted_race_times = model.predict(X)
merged_data["PredictedRaceTime (s)"] = predicted_race_times
race_predictions_2026 = merged_data.sort_values(by="PredictedRaceTime (s)")

In [14]:
# Evaluate
y_pred = model.predict(X_test)
print(f"\n Model Error (MAE): {mean_absolute_error(y_test, y_pred):.2f} seconds")


 Model Error (MAE): 0.28 seconds


In [19]:
rf = RandomForestRegressor(n_estimators=200, random_state=38)
rf.fit(X_train, y_train)
rf_y_pred = rf.predict(X_test)
print(f"\n Model Error (MAE): {mean_absolute_error(y_test, rf_y_pred):.2f} seconds")


 Model Error (MAE): 0.19 seconds


### Predicted vs Actual Result (2026 Italian GP)

In [15]:
# Actual 2026 Italian GP race result (formula1.com/en/results/2026/races/1293/italy/race-result)
actual_results_2026 = pd.DataFrame({
    "Driver": ["ANT", "RUS", "VER", "NOR", "PIA", "HAM", "GAS", "LIN", "COL", "TSU",
               "BOR", "HUL", "SAI", "LAW", "BEA", "OCO", "ALB", "PER", "BOT",
               "STR", "ALO", "LEC"],
    "ActualPosition": list(range(1, 23)),
    "Status": ["Finished"] * 19 + ["DNF (26 laps)", "DNF (23 laps)", "DNF (1 lap)"]
})

In [16]:
# Predicted finishing order: rank drivers by predicted average lap time, fastest first.
# ActualAvgLapTime (s) comes from the real race laps loaded via FastF1 (merged_data),
# not the results page, since lapped cars there only show "+1 lap" / "+2 laps", not a time.
predicted_ranking = race_predictions_2026[["Driver", "LapTime (s)", "PredictedRaceTime (s)"]].reset_index(drop=True)
predicted_ranking = predicted_ranking.rename(columns={"LapTime (s)": "ActualAvgLapTime (s)"})
predicted_ranking["PredictedPosition"] = predicted_ranking.index + 1
predicted_ranking["TimeDiff (s)"] = predicted_ranking["ActualAvgLapTime (s)"] - predicted_ranking["PredictedRaceTime (s)"]

# Combine predicted and actual results. Left join on the actual results keeps LEC
# in the table even though the model couldn't predict him (no clean lap data to
# train/predict on after his lap-1 DNF) -- he just gets NaN predicted/actual values.
results_comparison = actual_results_2026.merge(predicted_ranking, on="Driver", how="left")
results_comparison["PositionDiff"] = results_comparison["ActualPosition"] - results_comparison["PredictedPosition"]

# Order from P1 down to last classified position
results_comparison = results_comparison[
    ["Driver", "ActualPosition", "PredictedPosition", "PositionDiff",
     "ActualAvgLapTime (s)", "PredictedRaceTime (s)", "TimeDiff (s)", "Status"]
].sort_values("ActualPosition").reset_index(drop=True)
results_comparison

,Driver,ActualPosition,PredictedPosition,PositionDiff,ActualAvgLapTime (s),PredictedRaceTime (s),TimeDiff (s),Status
0,ANT,1,7.0,-6.0,88.502720,88.502720,-1.475683e-07,Finished
1,RUS,2,1.0,1.0,87.052020,87.052020,-6.551726e-08,Finished
2,VER,3,2.0,1.0,87.391429,87.391429,-3.864383e-07,Finished
3,NOR,4,4.0,0.0,87.618224,87.618224,3.106585e-07,Finished
4,PIA,5,3.0,2.0,87.617571,87.617572,-7.616255e-08,Finished
5,HAM,6,6.0,0.0,87.810612,87.810611,8.026618e-07,Finished
6,GAS,7,5.0,2.0,87.620102,87.620102,-3.843119e-07,Finished
7,LIN,8,10.0,-2.0,88.175122,88.679816,-5.046933e-01,Finished
8,COL,9,11.0,-2.0,87.930490,88.679926,-7.494361e-01,Finished
9,TSU,10,8.0,2.0,88.431816,88.521384,-8.956764e-02,Finished


## Additional model